In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import seaborn as sns


In [8]:
dataset=pd.read_csv('Final_CSCI323_DATASET.csv')

In [ ]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71993 entries, 0 to 71992
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Age               71993 non-null  int64  
 1   Sex               71993 non-null  object 
 2   BMI               71993 non-null  float64
 3   DiabetesType      71993 non-null  object 
 4   HbA1c             71993 non-null  float64
 5   ActivityLevel     71993 non-null  object 
 6   CalorieNeeds      71993 non-null  int64  
 7   CarbTolerance     71993 non-null  int64  
 8   SugarSensitivity  71993 non-null  object 
 9   Breakfast_Weekly  71993 non-null  object 
 10  Lunch_Weekly      71993 non-null  object 
 11  Dinner_Weekly     71993 non-null  object 
 12  Snacks_Weekly     71993 non-null  object 
 13  Drinks_Weekly     71993 non-null  object 
dtypes: float64(2), int64(3), object(9)
memory usage: 7.7+ MB


In [10]:
dataset.head()


,Age,Sex,BMI,DiabetesType,HbA1c,ActivityLevel,CalorieNeeds,CarbTolerance,SugarSensitivity,Breakfast_Weekly,Lunch_Weekly,Dinner_Weekly,Snacks_Weekly,Drinks_Weekly
0,0.487840,F,1.037247,Type 1,-0.305549,Moderate,0.077033,-0.997273,High,"['Chickpea Flour Flatbread', 'Oat Porridge (no...","['Cabbage Rolls', 'Stuffed Peppers (veggie)', ...","['Grilled Fish', 'Bamya', 'Arabic Salad', 'Mus...","['Low-Fat Yogurt', 'Yogurt & Cucumber Dip', 'S...","['Laban', 'Water', 'Saffron Milk', 'Camel Milk..."
1,1.238363,F,-0.126514,Type 1,0.145896,Low,-1.439241,0.431960,Medium,"['Baidh Wa Tomat', 'Egg & Spinach Wrap', 'Chic...","['Stuffed Peppers (veggie)', 'Lentil Patties',...","['Kale & Lentil Soup', 'Okra Stew', 'Tabbouleh...","['Yogurt & Cucumber Dip', 'Yogurt & Cucumber D...","['Cucumber Mint Laban', 'Saffron Milk', 'Water..."
2,-0.089486,F,-0.016428,Type 1,-1.358920,Low,-0.961511,0.547221,Medium,"['Balaleet (reduced sugar)', 'Zaatar Sandwich ...","['Madrooba (Fish)', 'Madrooba (Fish)', 'Harees...","['Pumpkin Soup', 'Barley Soup', 'Barley Soup',...","['', 'Fruit Salad (low GI fruits)', 'Fruit Sal...","['', '', '', '', '', '', '']"
3,-0.897743,M,1.273145,Type 1,0.296377,High,1.821786,-1.688837,Medium,"['Oat Porridge (no sugar)', 'Baidh Wa Tomat', ...","['Chickpea Salad', 'Lentil Patties', 'Falafel ...","['Grilled Fish', 'Grilled Halloumi Salad', 'Ka...","['Labneh & Olives', 'Boiled Chickpeas', 'Sweet...","['Sulaimani Tea', 'Rose Laban', 'Laban', 'Cucu..."
4,0.718770,M,-0.032155,Type 1,-1.434161,High,1.240202,0.063126,Medium,"['Whole Wheat Chami', 'Whole Wheat Chami', 'Za...","['Vegetarian Majboos', 'Madrooba (Fish)', 'Har...","['Barley Soup', 'Barley Soup', 'Harees (formal...","['Boiled Corn', 'Boiled Corn', 'Boiled Corn', ...","['Date Smoothie', 'Date Smoothie', 'Date Smoot..."


In [13]:
import pandas as pd
import ast
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os

# Load dataset
df = dataset.copy()

# ==============================
# A. PATIENT FEATURES CLEANING
# ==============================

# Normalize numeric columns
numeric_cols = ["Age", "BMI", "HbA1c", "CalorieNeeds", "CarbTolerance"]
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Encode categorical columns
categorical_cols = ["Sex", "DiabetesType", "ActivityLevel", "SugarSensitivity"]
for col in categorical_cols:
    df[col] = df[col].str.strip().str.lower()  # standardize
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# ==============================
# B. FOOD TEXT CLEANING
# ==============================

food_cols = ["Breakfast_Weekly", "Lunch_Weekly", "Dinner_Weekly", "Snacks_Weekly", "Drinks_Weekly"]

def clean_food_list(food_str):
    """Convert stringified list to clean list of foods."""
    try:
        # Convert string to Python list
        foods = ast.literal_eval(food_str)
        if not isinstance(foods, list):
            return []
        # Lowercase, strip, remove empty
        foods = [f.strip().lower() for f in foods if f and f.strip()]
        # Remove duplicates
        foods = list(set(foods))
        return foods
    except (ValueError, SyntaxError):
        return []

for col in food_cols:
    df[col] = df[col].apply(clean_food_list)

# ==============================
# C. CREATE FOOD DICTIONARY (LONG FORMAT)
# ==============================

food_dict_rows = []

for idx, row in df.iterrows():
    patient_id = idx  # use index as unique ID
    for meal in food_cols:
        for food in row[meal]:
            food_dict_rows.append({
                "PatientID": patient_id,
                "MealType": meal.replace("_Weekly", ""),
                "FoodItem": food
            })

food_df = pd.DataFrame(food_dict_rows)

# ==============================
# D. HANDLE MISSING VALUES
# ==============================

# Replace empty with "unknown"
food_df["FoodItem"].replace("", "unknown", inplace=True)

# Drop patients with no food records at all
valid_patients = food_df["PatientID"].unique()
df = df.loc[df.index.isin(valid_patients)]

# ==============================
# OUTPUT
# ==============================
# Save df (full dataset after cleaning)
df.to_csv("cleaned_dataset.csv", index=False)

# Save food_df (food-specific dataset)
food_df.to_csv("food_dataset.csv", index=False)
